<a href="https://colab.research.google.com/github/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/05_geri_yayilim_optimizasyon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a> &nbsp; <a href="https://github.com/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/05_geri_yayilim_optimizasyon.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-Kaynak-181717?logo=github" alt="GitHub'da Görüntüle"/></a>

> 🚀 **Bu notebook'u tarayıcıda çalıştırmak için sol üstteki "Colab'da Aç" butonuna tıklayın** — hiçbir şey kurmanıza gerek yok!


<div style="background: linear-gradient(135deg, #0B3D91 0%, #1565C0 60%, #1E88E5 100%); padding: 26px 30px; border-radius: 14px; color: white; margin: 0 0 24px 0; font-family: Georgia, serif; box-shadow: 0 4px 14px rgba(11,61,145,0.25);">

<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
  <span style="font-size: 11px; font-weight: bold; letter-spacing: 2px; color: #BBDEFB;">MEB · YEĞİTEK · DERİN ÖĞRENME SERİSİ</span>
  <span style="font-size: 11px; background: rgba(255,255,255,0.18); padding: 5px 12px; border-radius: 20px; color: white; font-weight: bold;">DERS 5 / 9 / 5</span>
</div>

<h1 style="margin: 8px 0 4px 0; color: white; font-size: 28px; line-height: 1.2;">🔄 Geri Yayılım ve Optimizasyon — Ağ Nasıl Öğrenir?</h1>

<p style="margin: 8px 0 0 0; color: #E3F2FD; font-size: 13px;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong> — Öğretmenler için Uygulamalı Yapay Zeka<br>
<span style="color: #BBDEFB;">⏱️ Süre: ~70 dakika · 🖥️ Ortam: Google Colab veya yerel Python</span>
</p>

<div style="border-top: 1px solid rgba(255,255,255,0.22); margin-top: 14px; padding-top: 12px; font-size: 12px; color: #E3F2FD;">
🎯 Bu notebook <strong>non-teknik kitle</strong> için hazırlanmıştır · Kavramlar günlük hayattan analojilerle anlatılır · Kod minimum, açıklama maksimumdur.
</div>

</div>

## 🎯 Bu Notebook'ta Ne Öğreneceğiz?

İleri yayılım: veri girer, tahmin çıkar ✅
Loss: tahmin ne kadar yanlış sayıyla biliyoruz ✅
**Şimdi en kritik adım:** Bu yanlışı azaltacak şekilde **ağırlıkları nasıl güncelleyeceğiz?**

Cevap: **Geri Yayılım (Backpropagation)** + **Optimizasyon**.

Bu notebook, serinin **en zorlu** ama **en sihirli** bölümü:

- 🔗 **Zincir kuralı (Chain Rule)** — kavramsal, sezgisel anlatım
- ⬅️ Hatayı **geriye doğru** yayma — her ağırlık ne kadar suçlu?
- 💪 NumPy ile **2 katmanlı XOR ağını sıfırdan eğitme**
- ⚙️ **Optimizer**'lar: SGD, Momentum, Adam — farkları
- 🆚 Keras ile aynı ağ → 5 satır
- 🎯 **Tüm seri burada birleşiyor!**

## 🤔 Önce Bir Senaryo

Bir mutfakta yemek bozuldu 🍲. Şefiniz dedi ki:

> "Yemek tuzlu çıktı. **Sebebi kim?**"

- 1. Aşamada bahçıvan domatesi tuzlu mu yetiştirdi?
- 2. Aşamada manav fazla mı satıyor?
- 3. Aşamada aşçı yemeğe fazla mı tuz ekledi?
- 4. Aşamada garson masada üzerine ekti mi?

**Suç dağıtılmalı.** Ve dağıtım **sondan başa** doğru yapılır: sondaki şüpheli en açık görülen, oradan geriye doğru gidilir.

Geri yayılım da budur:

> 🎓 **Hata sondaki tahminden başlar, her katmanın ne kadar "suçlu" olduğu hesaplanır, ve ağırlıklar buna göre güncellenir.**


## 1️⃣ Zincir Kuralı — Sezgisel

Diyelim ki bir basketbol takımının skoru üç şeye bağlı:

> Skor = f(antrenör'ün stratejisi → oyuncuların performansı → topun atışı)

Antrenör küçük bir karar değiştirdi. Skoru ne kadar etkiler? Cevap **zincirleme** etkilerin çarpımıdır:

$$\frac{d\text{Skor}}{d\text{Antrenör}} = \frac{d\text{Skor}}{d\text{Atış}} \times \frac{d\text{Atış}}{d\text{Performans}} \times \frac{d\text{Performans}}{d\text{Antrenör}}$$

> 🎓 **Türev burada "küçük değişimi" demek.** Her aşamadaki "ne kadar etkiler"i çarpıyoruz.

Sinir ağında da aynısı:

$$\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial a_2} \times \frac{\partial a_2}{\partial z_2} \times \frac{\partial z_2}{\partial a_1} \times \frac{\partial a_1}{\partial z_1} \times \frac{\partial z_1}{\partial W_1}$$

Bu **zincir kuralı**. Korkutucu görünür ama her parça **küçük ve kolay** hesaplanır.


## 📦 Kurulum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
print("✅ Hazır!")

## 2️⃣ Tek Nöronda Geri Yayılım — Hızlıca Tekrar

Notebook 01'de yapmıştık. Tek nöron için:

$$y = w \cdot x + b, \quad L = (y - y_{\text{gerçek}})^2$$

Türevler:

$$\frac{\partial L}{\partial w} = 2(y - y_{\text{gerçek}}) \cdot x, \quad \frac{\partial L}{\partial b} = 2(y - y_{\text{gerçek}})$$

Güncelleme:

$$w_{\text{yeni}} = w - \eta \cdot \frac{\partial L}{\partial w}$$

$\eta$ (eta) = öğrenme oranı.

## 3️⃣ Çok Katmanlı Ağda Geri Yayılım

Şimdi 2 katmanlı bir ağ yazalım — **XOR'u sıfırdan eğiteceğiz**.

```
[2 girdi] → [Dense(4) + ReLU] → [Dense(1) + Sigmoid] → [Loss: BCE]
```

İleri yayılım (notebook 03'ten):

$$z_1 = W_1 x + b_1, \quad a_1 = \text{ReLU}(z_1)$$
$$z_2 = W_2 a_1 + b_2, \quad a_2 = \sigma(z_2)$$

Geri yayılım — **sondan başa**:

| Türev | Formül |
|---|---|
| $\partial L / \partial z_2$ | $a_2 - y$  *(BCE + Sigmoid kombinasyonu için sade)* |
| $\partial L / \partial W_2$ | $\partial L / \partial z_2 \cdot a_1^T$ |
| $\partial L / \partial b_2$ | $\partial L / \partial z_2$ |
| $\partial L / \partial a_1$ | $W_2^T \cdot \partial L / \partial z_2$ |
| $\partial L / \partial z_1$ | $\partial L / \partial a_1 \cdot \text{ReLU}'(z_1)$ |
| $\partial L / \partial W_1$ | $\partial L / \partial z_1 \cdot x^T$ |
| $\partial L / \partial b_1$ | $\partial L / \partial z_1$ |

> 💡 **Korkmayın!** Aşağıda kodda her satır markdown ile açıklanacak.

In [ ]:
# Aktivasyonlar ve türevleri
def sigmoid(x): return 1 / (1 + np.exp(-x))
def relu(x): return np.maximum(0, x)
def relu_turev(x): return (x > 0).astype(float)

# XOR verisi
X = np.array([[0,0,1,1],
              [0,1,0,1]])    # şekil (2, 4)
Y = np.array([[0, 1, 1, 0]])  # şekil (1, 4)

# Ağırlıkları rastgele başlat (küçük değerler)
W1 = np.random.randn(4, 2) * 0.5
b1 = np.zeros((4, 1))
W2 = np.random.randn(1, 4) * 0.5
b2 = np.zeros((1, 1))

ogrenme_orani = 0.5
epoch_sayisi = 5000
loss_gecmisi = []

for epoch in range(epoch_sayisi):
    # ===== İLERİ YAYILIM =====
    Z1 = W1 @ X + b1            # (4, 4)
    A1 = relu(Z1)               # (4, 4)
    Z2 = W2 @ A1 + b2           # (1, 4)
    A2 = sigmoid(Z2)            # (1, 4)

    # ===== LOSS =====
    eps = 1e-12
    loss = -np.mean(Y * np.log(A2 + eps) + (1 - Y) * np.log(1 - A2 + eps))
    loss_gecmisi.append(loss)

    # ===== GERİ YAYILIM =====
    m = X.shape[1]   # batch büyüklüğü = 4

    dZ2 = A2 - Y                                        # (1, 4)
    dW2 = (1/m) * dZ2 @ A1.T                            # (1, 4)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)    # (1, 1)

    dA1 = W2.T @ dZ2                                    # (4, 4)
    dZ1 = dA1 * relu_turev(Z1)                          # (4, 4)
    dW1 = (1/m) * dZ1 @ X.T                             # (4, 2)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)    # (4, 1)

    # ===== GÜNCELLEME (SGD) =====
    W1 -= ogrenme_orani * dW1
    b1 -= ogrenme_orani * db1
    W2 -= ogrenme_orani * dW2
    b2 -= ogrenme_orani * db2

    if epoch % 1000 == 0:
        print(f"Epoch {epoch:>5} | Loss: {loss:.4f}")

print(f"\n🎉 Eğitim bitti! Son loss: {loss:.6f}")

### Test: XOR'u Çözebiliyor mu?

In [ ]:
# Eğitilmiş ağırlıklarla tahmin
Z1 = W1 @ X + b1; A1 = relu(Z1)
Z2 = W2 @ A1 + b2; A2 = sigmoid(Z2)

print("Girdi (x1,x2) | Gerçek | Tahmin")
print("-" * 38)
for i in range(4):
    print(f"   ({X[0,i]}, {X[1,i]})       |   {int(Y[0,i])}    | {A2[0,i]:.4f}")

print("\n✅ NumPy ile sıfırdan eğitilmiş ağ XOR'u çözdü!")

### Loss'un Düşüşünü Görelim

In [ ]:
plt.plot(loss_gecmisi, color='#E53935', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('📉 Hatanın Eğitim Boyunca Düşüşü', fontweight='bold')
plt.show()

print("👀 Başlangıçta yüksek olan hata, geri yayılım sayesinde düştü.")
print("   Her epoch'ta ağırlıklar 'biraz daha doğru' hale geldi.")

## 4️⃣ Optimizasyon — Sadece SGD Yetmez

Az önce kullandığımız:

$$w \leftarrow w - \eta \cdot \nabla L$$

Bu **vanilla SGD** (Stochastic Gradient Descent). Çalışıyor ama **yavaş** olabilir, **dar boğazlarda** sallanır.

### Üç popüler optimizer:

#### 1️⃣ SGD (Vanilla)
- Her adımda sadece o anki gradyana bakar
- "Sarhoş gibi" zikzak yapabilir
- En basit ama en yavaş

#### 2️⃣ Momentum
- Önceki adımlardan **momentum** taşır
- Top yokuştan aşağı **giderek hızlanır**
- $v \leftarrow \beta v + \nabla L$, sonra $w \leftarrow w - \eta v$
- Daha az salınım, daha hızlı yakınsama

> 🎓 **Analoji:** Ağır bir top — küçük çukurlarda takılmaz, momentumla geçer.

#### 3️⃣ Adam (Adaptive Moment Estimation)
- Her ağırlık için **ayrı öğrenme oranı**
- Hem momentum hem **adaptif ölçek** içerir
- **Bugünkü standart** (`optimizer='adam'`)

> 🎓 **Analoji:** Hem hızlı hem akıllı — eğri yerlerde yavaşlar, düz yerlerde hızlanır.

In [ ]:
# Üç optimizer kıyası — basit bir 1D problemde
# y = (x - 3)^2'nin minimumunu bulmaya çalış (cevap x=3)

def gradient(x):
    return 2 * (x - 3)

def egitim_simulasyonu(optimizer, x0=10.0, lr=0.1, adim=30):
    x = x0
    yol = [x]
    if optimizer == 'sgd':
        for _ in range(adim):
            x = x - lr * gradient(x)
            yol.append(x)
    elif optimizer == 'momentum':
        v = 0; beta = 0.9
        for _ in range(adim):
            v = beta * v + gradient(x)
            x = x - lr * v
            yol.append(x)
    elif optimizer == 'adam':
        m = 0; v = 0; beta1 = 0.9; beta2 = 0.999; eps = 1e-8
        for t in range(1, adim+1):
            g = gradient(x)
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * g**2
            m_hat = m / (1 - beta1**t)
            v_hat = v / (1 - beta2**t)
            x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
            yol.append(x)
    return yol

sgd_yol = egitim_simulasyonu('sgd')
mom_yol = egitim_simulasyonu('momentum')
adam_yol = egitim_simulasyonu('adam', lr=1.0)

plt.plot(sgd_yol, label='SGD', linewidth=2, marker='o', markersize=3)
plt.plot(mom_yol, label='Momentum', linewidth=2, marker='s', markersize=3)
plt.plot(adam_yol, label='Adam', linewidth=2, marker='^', markersize=3)
plt.axhline(y=3, color='green', linestyle='--', label='Hedef (x=3)')
plt.xlabel('Adım'); plt.ylabel('x')
plt.title('🏎️ Üç Optimizer\'ın Hedefe Yaklaşma Hızı', fontweight='bold')
plt.legend()
plt.show()

print(f"30 adım sonra:")
print(f"  SGD     : x = {sgd_yol[-1]:.4f}")
print(f"  Momentum: x = {mom_yol[-1]:.4f}")
print(f"  Adam    : x = {adam_yol[-1]:.4f}")

## 🆚 Karma Demo: Aynı Şeyi Keras ile

Yukarıda **40+ satır kod** yazdık (ileri + geri + güncelleme). Keras'ta **5 satır**:

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# XOR verisi (Keras formatında: örnek_sayısı × özellik)
X_keras = np.array([[0,0],[0,1],[1,0],[1,1]], dtype='float32')
Y_keras = np.array([[0],[1],[1],[0]], dtype='float32')

# 5 satır
model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(4, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Eğit
model.fit(X_keras, Y_keras, epochs=2000, verbose=0)

# Test
tahminler = model.predict(X_keras, verbose=0)
print("Keras ile XOR çözümü:")
for i in range(4):
    print(f"  ({X_keras[i,0]}, {X_keras[i,1]}) → {tahminler[i,0]:.4f}")

print("\n💡 Keras arka planda BİZİM yazdığımızın AYNISINI yapıyor:")
print("   - İleri yayılım (matris çarpımları + aktivasyon)")
print("   - Loss hesabı")
print("   - Geri yayılım (otomatik türev)")
print("   - Adam ile ağırlık güncellemesi")
print("\n   Sadece bizden gizliyor. Şimdi içinde ne döndüğünü BİLİYORUZ.")

## 🎓 Anahtar Kavramlar

| Terim | Türkçe | Anlamı |
|---|---|---|
| **Forward Propagation** | İleri Yayılım | Veri → Katmanlar → Tahmin |
| **Loss** | Kayıp | Tahmin ne kadar yanlış (sayı) |
| **Backpropagation** | Geri Yayılım | Hata → Katmanlar (geri) → Gradyanlar |
| **Chain Rule** | Zincir Kuralı | Türevlerin çarpımı |
| **Gradient** | Gradyan | Kayıp fonksiyonunun ağırlığa göre eğimi |
| **Optimizer** | Eniyileyici | Gradyana göre nasıl güncelleme yapılır |
| **Learning Rate** | Öğrenme Oranı | Adım boyutu (η) |
| **Epoch** | Tur | Tüm verinin bir kez gözden geçirilmesi |

### 🎓 Anahtar İçgörü

> **Sinir ağı 4 adımda öğrenir:**
> 1. **İleri yayılım** → tahmin yap
> 2. **Loss hesapla** → ne kadar yanlışsın?
> 3. **Geri yayılım** → suç dağıt (gradyanlar)
> 4. **Güncelle** → ağırlıkları biraz düzelt
>
> Tekrarla, tekrarla, tekrarla...

### 🚀 Şimdi Ne Olacak?

**Tüm temelleri bitirdik!** Bundan sonraki notebook'lar **Keras ile** yapılacak — çünkü artık:
- Keras'ın arka planda **ne yaptığını biliyoruz**
- Hangi loss / aktivasyon / optimizer'ı **niye seçtiğimizi anlıyoruz**
- Bir sorun çıkarsa **nereye bakacağımızı biliyoruz**

Sıradaki: **Iris çiçek sınıflandırma** — yeni öğrendiklerimizi gerçek bir veri setinde uygulayacağız! 🌸

<div style="background: #0B3D91; color: #BBDEFB; padding: 26px 30px; border-radius: 14px; margin-top: 32px; font-family: Georgia, serif;">

<h3 style="color: white; margin: 0 0 12px 0; border-bottom: 2px solid #FFC107; padding-bottom: 8px; display: inline-block;">MEB · YEĞİTEK</h3>

<p style="font-family: Calibri, sans-serif; font-size: 14px; color: #E3F2FD; margin: 0 0 8px 0; line-height: 1.6;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong><br>
Öğretmen ve eğitimciler için uygulamalı derin öğrenme eğitim serisi.
</p>

<p style="font-family: Calibri, sans-serif; font-size: 12px; color: #BBDEFB; margin: 12px 0 0 0;">
🎓 Bu notebook eğitim amaçlıdır · Kullanılan tüm kütüphaneler açık kaynaktır.
</p>

<div style="background: rgba(255,255,255,0.08); border-left: 3px solid #FFC107; padding: 12px 16px; margin-top: 16px; font-family: Calibri, sans-serif; font-size: 13px; color: #FFF9E6;">
➡️ <strong>Bir sonraki notebook:</strong> <code>06_iris_sinir_agi.ipynb</code> — Iris çiçek sınıflandırma — Keras ile pratiğe dönüş</div>
<p style="font-family: Calibri, sans-serif; font-size: 11px; color: #90CAF9; margin: 14px 0 0 0; text-align: center; border-top: 1px solid rgba(255,255,255,0.15); padding-top: 12px;">
© 2026 MEB YEĞİTEK · Derin Öğrenme Serisi
</p>

</div>